In [20]:
import requests
import pandas as pd
import time

API_KEY = ""

PLACES_TEXT_SEARCH_URL = "https://maps.googleapis.com/maps/api/place/textsearch/json"
PLACE_DETAILS_URL = "https://maps.googleapis.com/maps/api/place/details/json"

def get_place_details(place_id):
    params = {
        "place_id": place_id,
        "fields": "name,formatted_address,formatted_phone_number,international_phone_number,website",
        "key": API_KEY
    }
    response = requests.get(PLACE_DETAILS_URL, params=params).json()
    return response.get("result", {})

def extract_places(keywords, location_name):
    records = []
    seen_place_ids = set()

    for keyword in keywords:
        query = f"{keyword} in {location_name}"
        print(f"Searching for: {query}")

        params = {
            "query": query,
            "key": API_KEY
        }

        while True:
            response = requests.get(PLACES_TEXT_SEARCH_URL, params=params).json()
            results = response.get("results", [])

            for place in results:
                place_id = place["place_id"]
                if place_id in seen_place_ids:
                    continue
                seen_place_ids.add(place_id)

                details = get_place_details(place_id)
                records.append({
                    "name": details.get("name"),
                    "address": details.get("formatted_address"),
                    "phone_local": details.get("formatted_phone_number"),
                    "phone_international": details.get("international_phone_number"),
                    "website": details.get("website"),
                    "email": None  # Emails not available from Google
                })

                # Rate limiting to avoid quota issues
                time.sleep(0.1)

            next_page_token = response.get("next_page_token")
            if not next_page_token:
                break
            params["pagetoken"] = next_page_token
            time.sleep(2)  # Required delay for next_page_token

    return pd.DataFrame(records)


In [22]:
keywords = ["pharmacy", "chemist", "medical store"]
location_name = "Naubasta, Kanpur, Uttar Pradesh, India"

df = extract_places(keywords, location_name)
print(df.head())

Searching for: pharmacy in Naubasta, Kanpur, Uttar Pradesh, India
Searching for: chemist in Naubasta, Kanpur, Uttar Pradesh, India
Searching for: medical store in Naubasta, Kanpur, Uttar Pradesh, India
                                name  \
0    Apollo Pharmacy Naubasta Kanpur   
1                      Rahul Chemist   
2    Sri Maa Chemist (Medical store)   
3  Zeelab Pharmacy - Naubasta Kanpur   
4                       A R PHARMACY   

                                             address    phone_local  \
0  Ground Floor, No. 1186WZ, Vasant Vihar, Naubas...  079 4284 5139   
1  Y BLOCK, 128/983, Hamirpur Rd, Thana Naubasta,...   063893 13313   
2  128/983 Y-Block kidwai nagar, near by paunjab ...   091989 47532   
3  Shop No-01, House No--30, Hamirpur Rd, Naubast...   098962 78230   
4  131/4, Yogendra Vihar Khandepur Rd, Naubasta, ...   088088 15730   

  phone_international                                            website email  
0    +91 79 4284 5139  https://www.apollopharmacy

In [23]:
df

,name,address,phone_local,phone_international,website,email
0,Apollo Pharmacy Naubasta Kanpur,"Ground Floor, No. 1186WZ, Vasant Vihar, Naubas...",079 4284 5139,+91 79 4284 5139,https://www.apollopharmacy.in/?utm_source=gmb&...,None
1,Rahul Chemist,"Y BLOCK, 128/983, Hamirpur Rd, Thana Naubasta,...",063893 13313,+91 63893 13313,None,None
2,Sri Maa Chemist (Medical store),"128/983 Y-Block kidwai nagar, near by paunjab ...",091989 47532,+91 91989 47532,None,None
3,Zeelab Pharmacy - Naubasta Kanpur,"Shop No-01, House No--30, Hamirpur Rd, Naubast...",098962 78230,+91 98962 78230,https://zeelabpharmacy.com/,None
4,A R PHARMACY,"131/4, Yogendra Vihar Khandepur Rd, Naubasta, ...",088088 15730,+91 88088 15730,None,None
...,...,...,...,...,...,...
113,Ashu Medical Store,"20A, Machchharia Chouraha, Naubasta, Kanpur, U...",094501 25657,+91 94501 25657,None,None
114,Ghanshyam Medical Store,"Purani Basti, Naubasta, Kanpur, Uttar Pradesh ...",099356 08519,+91 99356 08519,None,None
115,P n Sharma medical store,"1362/B, Baudh Vihar, Naubasta, Kanpur, Uttar P...",None,None,None,None
116,Bala Ji Medical Store,"1190, W-2, Vasant Vihar, Naubasta, Kanpur, Utt...",090268 66785,+91 90268 66785,None,None
